# 02 — Build v3 training windows

Build gap-aware 48-hour inputs and their exact PM2.5 target 24 hours later. The August 2026 smoke period is kept completely untouched for evaluation.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd()
ROOT = ROOT if (ROOT / 'common').exists() else ROOT.parent
sys.path.insert(0, str(ROOT / 'modeling'))

import numpy as np
import pandas as pd

from splits import time_split
from windowing import FEATURE_COLS, build_supervised

VALIDATION_START = '2025-07-01'
SMOKE_TEST_START = '2026-08-01'

df = pd.read_parquet(ROOT / 'modeling' / 'feat_airquality.parquet')
df['valid_time'] = pd.to_datetime(df['valid_time'], utc=True)

X, y, meta = build_supervised(df)
train, validation, test = time_split(
    meta,
    validation_start=VALIDATION_START,
    test_start=SMOKE_TEST_START,
)

pm25_index = FEATURE_COLS.index('pm25')
current_pm25 = X[:, -1, pm25_index].astype(np.float32)

assert len(train) and len(validation) and len(test)
assert meta['valid_time'].iloc[train].max() < pd.Timestamp(VALIDATION_START, tz='UTC')
assert meta['valid_time'].iloc[validation].max() < pd.Timestamp(SMOKE_TEST_START, tz='UTC')

def describe_split(name, indexes):
    target = y[indexes]
    current = current_pm25[indexes]
    high = target > 35.4
    severe = target > 55.4
    onset = high & (current <= 12.0)
    print(
        f'{name:<12} n={len(indexes):,} '
        f'high={int(high.sum()):,} '
        f'severe={int(severe.sum()):,} '
        f'onset={int(onset.sum()):,}'
    )

describe_split('train', train)
describe_split('validation', validation)
describe_split('smoke test', test)

output = ROOT / 'modeling' / 'artifacts' / 'v3'
output.mkdir(parents=True, exist_ok=True)

np.savez_compressed(
    output / 'windows.npz',
    X=X,
    y=y,
    current=current_pm25,
    train=train,
    validation=validation,
    test=test,
    station=meta['station_id'].to_numpy(dtype=str),
)

print('shape:', X.shape, '| features:', len(FEATURE_COLS))
print('saved:', output / 'windows.npz')